# เทรนโมเดล text-to-SQL ภาษาไทย (รันบน Colab)

ก่อนรัน ให้อัปโหลดไฟล์เหล่านี้เข้า `/content/` (ลาก-วาง หรือใช้แผงไฟล์ด้านซ้ายของ Colab):
- `ml/format.py`
- `ml/data/train.jsonl`
- `ml/data/val.jsonl`

ตั้งค่า Runtime -> Change runtime type -> GPU (T4) ก่อนรันเซลล์ถัดไป

ค่าที่ต้องจดลง `docs/MODEL_CARD.md` จะถูกพิมพ์ออกมาท้ายโน้ตบุ๊ก (เวลาเทรน, GPU, hyperparameters, จำนวนข้อมูล)

In [ ]:
!pip install -q -U transformers accelerate peft datasets

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

# โฟลเดอร์ปลายทางสำหรับบันทึก checkpoint และโมเดลที่ export แล้ว
DRIVE_OUT_DIR = '/content/drive/MyDrive/thamdi_model_v1'

In [ ]:
import json
import sys
import time

import torch

sys.path.insert(0, '/content')
from format import build_prompt

# --- ตั้งค่าหลัก: เปลี่ยน MODEL_NAME ได้ตามที่เลือก ---
CONFIG = {
    'model_name': 'Qwen/Qwen2.5-0.5B-Instruct',
    'use_lora': True,
    'lora_r': 16,
    'lora_alpha': 32,
    'lora_dropout': 0.05,
    'max_length': 512,
    'learning_rate': 2e-4,
    'num_train_epochs': 3,
    'per_device_train_batch_size': 8,
    'gradient_accumulation_steps': 2,
    'seed': 42,
}
print(CONFIG)

In [ ]:
def load_jsonl(path):
    records = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

train_records = load_jsonl('/content/train.jsonl')
val_records = load_jsonl('/content/val.jsonl')
print(f'train: {len(train_records)} ข้อ, val: {len(val_records)} ข้อ')

def by_category(records):
    counts = {}
    for r in records:
        counts[r['category']] = counts.get(r['category'], 0) + 1
    return counts

print('train ต่อหมวด:', by_category(train_records))
print('val ต่อหมวด:', by_category(val_records))

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    CONFIG['model_name'],
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    device_map='auto',
)

if CONFIG['use_lora']:
    from peft import LoraConfig, get_peft_model
    lora_config = LoraConfig(
        r=CONFIG['lora_r'],
        lora_alpha=CONFIG['lora_alpha'],
        lora_dropout=CONFIG['lora_dropout'],
        task_type='CAUSAL_LM',
        target_modules='all-linear',
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

In [ ]:
# ต่อ prompt (build_prompt เดียวกับที่ preprocess.py ใช้) กับคำตอบ แล้ว mask ส่วน prompt ออกจาก loss
# ให้โมเดลเรียนรู้แค่การเขียนคำตอบต่อจาก "### SQL" เท่านั้น ไม่ต้องเรียนรู้การทำนาย schema/คำถามซ้ำ

def encode_example(record, tokenizer, max_length):
    prompt = build_prompt(record['question'], record['schema_ddl'])
    completion = record['target'] + tokenizer.eos_token

    prompt_ids = tokenizer(prompt, add_special_tokens=False)['input_ids']
    completion_ids = tokenizer(completion, add_special_tokens=False)['input_ids']

    input_ids = (prompt_ids + completion_ids)[:max_length]
    labels = ([-100] * len(prompt_ids) + completion_ids)[:max_length]
    attention_mask = [1] * len(input_ids)
    return {'input_ids': input_ids, 'attention_mask': attention_mask, 'labels': labels}


class SqlDataset(torch.utils.data.Dataset):
    def __init__(self, records, tokenizer, max_length):
        self.records = records
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        return encode_example(self.records[idx], self.tokenizer, self.max_length)


def collate(batch, pad_token_id):
    max_len = max(len(x['input_ids']) for x in batch)
    input_ids, attention_mask, labels = [], [], []
    for x in batch:
        pad_len = max_len - len(x['input_ids'])
        input_ids.append(x['input_ids'] + [pad_token_id] * pad_len)
        attention_mask.append(x['attention_mask'] + [0] * pad_len)
        labels.append(x['labels'] + [-100] * pad_len)
    return {
        'input_ids': torch.tensor(input_ids),
        'attention_mask': torch.tensor(attention_mask),
        'labels': torch.tensor(labels),
    }

train_dataset = SqlDataset(train_records, tokenizer, CONFIG['max_length'])

In [ ]:
from functools import partial

from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir='/content/checkpoints',
    num_train_epochs=CONFIG['num_train_epochs'],
    per_device_train_batch_size=CONFIG['per_device_train_batch_size'],
    gradient_accumulation_steps=CONFIG['gradient_accumulation_steps'],
    learning_rate=CONFIG['learning_rate'],
    logging_steps=20,
    save_strategy='epoch',
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    report_to=[],
    seed=CONFIG['seed'],
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=partial(collate, pad_token_id=tokenizer.pad_token_id),
)

start_time = time.time()
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
print('GPU:', gpu_name)

trainer.train()

train_seconds = time.time() - start_time
print(f'เทรนเสร็จใน {train_seconds / 60:.1f} นาที')

In [ ]:
import os

os.makedirs(DRIVE_OUT_DIR, exist_ok=True)

if CONFIG['use_lora']:
    merged_model = model.merge_and_unload()
else:
    merged_model = model

merged_model = merged_model.to(torch.float16)
merged_model.save_pretrained(f'{DRIVE_OUT_DIR}/model', safe_serialization=True)
tokenizer.save_pretrained(f'{DRIVE_OUT_DIR}/model')
print(f'บันทึกโมเดล (export ขนาดเล็ก, fp16) ไว้ที่ {DRIVE_OUT_DIR}/model')

In [ ]:
# วัดผลคร่าว ๆ บน val ด้วย exact match (เทียบ string ตรงตัวหลังตัด whitespace ส่วนเกิน)
# ไม่ใช่ตัวเลขที่ใช้รายงานจริง (ของจริงให้ Wissarut รันผ่าน eval/run.py) แต่ใช้เทียบระหว่างรอบเทรนได้

def normalize_sql(s):
    return ' '.join(s.strip().split())


@torch.no_grad()
def predict(question, schema_ddl, n=1, max_new_tokens=128):
    prompt = build_prompt(question, schema_ddl)
    inputs = tokenizer(prompt, return_tensors='pt').to(merged_model.device)
    outputs = merged_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        num_beams=max(n, 1),
        num_return_sequences=n,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )
    results = []
    for out in outputs:
        text = tokenizer.decode(out[inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        results.append(text.strip())
    return results


correct = 0
correct_by_category = {}
total_by_category = {}
sample_n = min(300, len(val_records))  # สุ่มตัวอย่างบางส่วนพอประเมินเร็ว ๆ ไม่ต้องวนครบทุกข้อ
for record in val_records[:sample_n]:
    pred = predict(record['question'], record['schema_ddl'], n=1)[0]
    cat = record['category']
    total_by_category[cat] = total_by_category.get(cat, 0) + 1
    if normalize_sql(pred) == normalize_sql(record['target']):
        correct += 1
        correct_by_category[cat] = correct_by_category.get(cat, 0) + 1

print(f'exact match บน val (สุ่ม {sample_n} ข้อ): {correct}/{sample_n} = {correct / sample_n:.1%}')
for cat, total in total_by_category.items():
    c = correct_by_category.get(cat, 0)
    print(f'  {cat}: {c}/{total} = {c / total:.1%}')

In [ ]:
print('=== ค่าที่ต้องจดลง docs/MODEL_CARD.md ===')
print('โมเดลตั้งต้น:', CONFIG['model_name'])
print('GPU:', gpu_name)
print(f'เวลาเทรน: {train_seconds / 60:.1f} นาที')
print('hyperparameters:', CONFIG)
print('จำนวนข้อมูลเทรน:', len(train_records), 'ต่อหมวด:', by_category(train_records))
print('จำนวนข้อมูล val:', len(val_records), 'ต่อหมวด:', by_category(val_records))